# Pod Readiness Probe Attack Reproduction

## 1. Mechanism overview

### 1.1 Attacking strategy: kubelet as a proxy

This attack implements strategy ❸ in Fig. 3 of the paper, described in §3.6:

> "a malicious container uses the kubelet as a proxy to send a large number of requests to interfaces that kube-apiserver does not allow to access. After that, to handle the incoming requests, the kube-apiserver interacts with other components and consumes resources considerably."

Table 1 of the paper lists the specific interface this relies on: Kubelet, `pods/status`, `patch`, "Send pod's status changes".

The mechanism works in two steps. First:

> "the kubelet synchronizes the pod's status changes to the kube-apiserver, and has patch verbs on pods/status interfaces [...] which containers' service account tokens usually cannot access. Thus, a malicious container can change its pod status and leverage the kubelet as a proxy to send patch verbs to the pods/status interfaces indirectly for the attack."

Second, the effect is amplified by data dependencies:

> "once the kube-apiserver receives pod status changes, it modifies the status of all the workload resources related to the pod, not just the pod status changes themselves. Thus, the kube-apiserver sends requests to the etcd component to update all these data structures [...] which causes the kube-apiserver and etcd to consume additional control plane resources vastly."

### 1.2 Case 3: pod readiness probe attack (§4.4, Fig. 6)

The paper's concrete instantiation of this strategy (§4.4):

> "malicious containers exploit the readiness probe to periodically change the status of the pods, causing the kubelet to synchronize these status changes to the kube-apiserver. After that, the kube-apiserver modifies all the workloads status related to the pods, and sends requests to the etcd to update all the data values. This step amplifies the effect of the attack, which makes the kube-apiserver and etcd consume up to 55.3% CPU resources."

For the first step, the readiness probe is used as the trigger:

> "a readiness probe checks the container's status periodically, and changes the pod's status to ready or not according to the check result. When the pod's status changes, the kubelet synchronizes the pod's status to the kube-apiserver periodically. Therefore, we can exploit the readiness probe to make the kubelet patch the pod's status to the kube-apiserver."

For the second step, the paper illustrates the amplification with the pod to replicaset to deployment dependency chain (Fig. 6):

> "the replicaset is a Kubernetes workload which is used to manage a group of pods, and the deployment is an upper-level workload that manages the replicasets. When a single pod status change is received by the kube-apiserver, it modifies the pod and related replicaset/deployment status, and updates all these values to the etcd back end. Thus, the data dependency [...] amplifies the pod status changes, which makes the kube-apiserver and etcd consume more CPU resources."

### 1.3 What the paper reports as impact

For the local environment, the paper reports (§4.4):

> "we run one hundred malicious containers, each of which is running in a pod managed by a separate replicaset and deployment. Each container leverages the readiness probe to modify its pods' ready status per second. As a result, kube-apiserver and etcd consume 55.3% additional CPU resources of the control plane."

Table 3 summarizes this as: "Readiness probe attack, strategy ❸, consuming control plane 55.3% CPU." This is the reference figure we compare our own measurements against in section 3.

## 2. Reproduction: manifests and methodology

### 2.1 Attack topology

Matching the paper's local experiment exactly, the reproduction uses 100 malicious containers, each running in its own pod, managed by its own separate ReplicaSet and Deployment, not a single Deployment with `replicas: 100`. This specific topology is required by the mechanism itself: the amplification described in §3.6 and §4.4 depends on each pod status change propagating through its own distinct replicaset/deployment chain down to etcd. A single shared Deployment with 100 replicas would create only one shared ReplicaSet/Deployment object pair, which does not match "each of which is running in a pod managed by a separate replicaset and deployment" (§4.4).

All 100 Deployments run in the `attacker-ns` namespace, pinned to `k8s-w1` via `nodeName`, consistent with the attacker-node isolation used for the other attacks.

### 2.2 Final manifest

Each Deployment runs a single-replica pod whose container's readiness probe is an `exec` check testing the parity of the current Unix timestamp (`test $(( $(date +%s) % 2 )) -eq 0`), run every second (`periodSeconds: 1`, `failureThreshold: 1`, `successThreshold: 1`). Since the check alternates between true and false on consecutive seconds, the pod's ready status flips roughly once per second, matching the paper's "modify its pods' ready status per second" (§4.4). The container itself does nothing but idle (`busybox:1.36`, `sleep infinity`); the readiness probe is the only active mechanism, consistent with the paper's framing of the readiness probe as the sole trigger for step one of the attack.

Each Deployment is generated with its own name (`readiness-flood-${i}`), its own pod template, and its own `matchLabels` selector, so that applying the manifest creates 100 independent Deployment/ReplicaSet/Pod chains rather than one Deployment scaled to 100 replicas, per the topology constraint in section 2.1. All pods are pinned to `k8s-w1` via `nodeName` and run under the same restricted security context as the other attacks (non-root, no privilege escalation, all capabilities dropped, `RuntimeDefault` seccomp profile), consistent with the `restricted` Pod Security Standard enforced on `attacker-ns`.

The manifest is produced by `generate-readiness-flood.sh`, which templates this Deployment definition `COUNT` times (default 100) into a single file, `readiness-flood-generated.yaml`.

### 2.3 Measurement scripts

**`generate-readiness-flood.sh`** generates the 100-Deployment manifest (`readiness-flood-generated.yaml`), one Deployment/ReplicaSet/Pod chain per malicious container, per the topology constraint in section 2.1.

**`launch-attack.sh`** applies the generated manifest and waits for the malicious pods to come up:

```bash
kubectl apply -f "$MANIFEST"
kubectl wait --for=condition=Ready pod -n "$NAMESPACE" -l "$LABEL" --timeout=120s || true
```

Cleanup is done with `kubectl delete deployment -n attacker-ns -l attack=readiness-flood`.

**`capture-cpu-loop-attack3.sh`** measures CPU usage of the kube-apiserver and etcd processes specifically, run on k8s-cp, matching the paper's own attribution for this attack ("kube-apiserver and etcd consume 55.3% additional CPU resources", §4.4). It reads `/proc/<pid>/stat` twice per interval, once at the start and once at the end, and computes the CPU-tick delta between the two for each process, the same method used for the CoreDNS and kube-apiserver attacks.

## 3. Results

Before/after averages, computed over 30 samples each (10s interval, 5 minutes), kube-apiserver and etcd process CPU specifically (`/proc/<pid>/stat` delta method):

In [1]:
from aggregate_averages_attack3 import find_csv_files, COLS, LABELS
import pandas as pd
from IPython.display import display, HTML

def load_combined(label):
    csv_files = find_csv_files(label, base_dir="../results")
    dfs = [pd.read_csv(f) for f in csv_files]
    return pd.concat(dfs, ignore_index=True)

before = load_combined("before")
after = load_combined("after")

rows = COLS
summary = pd.DataFrame({
    "Before": before[rows].mean().round(1),
    "After": after[rows].mean().round(1),
}).rename(index=LABELS)

display(HTML("<h2>Metrics comparison before / after attack</h2>"))
display(summary.style.set_properties(**{"font-size": "16pt"}).set_table_styles(
    [{"selector": "th", "props": [("font-size", "16pt")]}]
))

,Before,After
CPU kube-apiserver process (%),8.600000,70.200000
CPU etcd process (%),3.900000,23.100000
CPU kube-apiserver + etcd (%),12.500000,93.300000


| Metric | Before | After | Delta |
|---|---|---|---|
| CPU kube-apiserver process (%) | 8.6 | 70.2 | +61.6 |
| CPU etcd process (%) | 3.9 | 23.1 | +19.2 |
| CPU kube-apiserver + etcd (%) | 12.5 | 93.3 | +80.8 |

The measured additional CPU consumption on kube-apiserver and etcd combined (+80.8 percentage points) is higher than the paper's reported 55.3% for the local environment, but matches the paper's mechanism qualitatively: both kube-apiserver and etcd rise together under the attack, consistent with the pod to replicaset to deployment amplification chain described in §3.6 and §4.4. This is notably different from the kube-apiserver attack (Case 2, section attack2), where etcd CPU decreased rather than increased during the attack, a discrepancy not explained by the paper's description of that attack.